# TTM-Small-4M × EmbeddingGemma 2 — **Kế thừa 1.024 → 2.048 bước GPU (Checkpoint Bridge)**

**Đây là notebook đúng sau bản sửa đổi tiền đăng ký. KHÔNG dùng notebook GPU2048 cũ**, vì notebook cũ chạy lại 2.048 bước từ checkpoint gốc. Notebook này nạp **cả bốn checkpoint GPU1024 đã hoàn thành** cùng AdamW/RNG và chỉ tính **bước 1.025–2.048** cho A/B/C/D. Tổng số nhãn mới mỗi arm **1.024 × 4.096 = 4.194.304**. Tổng 2.048 bước/arm kể cả prefix cũ.

**Runtime:** Colab Tesla T4. Giữ `RESUME=False` lần đầu, `RESUME=True` chỉ khi tiếp tục từ một `READY` mới hợp lệ sau gián đoạn. Điểm lưu mới: 1.280, 1.536, 1.792, 2.048. Ngắt trước điểm READY đầu tiên của arm đang chạy thì dừng, không tự phát lại bước. Không đọc TEST, không chạy Residual/Gated.

**Điều kiện:** toàn bộ checkpoint/READY gốc, 1.024-step schedule prefix, SHA-256, AdamW step, RNG và môi trường Torch/CUDA/T4 **phải trùng** lượt cũ; nếu không, dừng trước `CLAIM`. Không tải lại EmbeddingGemma 2, không tải ZIP thủ công.

[Amendment đã khóa](https://github.com/minhtri22/TokenToModel-Lab/blob/24d0b7b1d5eea590ba1cfa0d00f74ffcbdd6f72a/docs/pilot/TTM_SMALL4M_GEMMA2_2048_CHECKPOINT_CONTINUATION_AMENDMENT_20261109.md). Thí nghiệm mở rộng đã sử dụng VAL trong việc quyết định ngân sách; kết quả sau cùng chỉ gọi exploratory. Kết thúc mới kiểm định độc lập, không tự động mở thực nghiệm khác.


In [ ]:
# Owner-amended GPU2048: inherit EXACT four GPU1024 AdamW/RNG states, train only 1025..2048.
RESUME = False  # Change to True ONLY to resume this exact run from SHA-verified READY.

import hashlib, subprocess, sys, urllib.request
from pathlib import Path
from google.colab import drive
import torch

RUN_ID = "ttm_small4m_gemma2_directswap_2048x4096_t4_gpu_20261109_v1"
BUILDER_COMMIT = "c50585dc7f58a1c377d26630d8234b2f6ce058eb"
BUILDER_SHA256 = "548491a1ab964bd7204d0e71773d47b524cb544b00ac80f6741fd90e722f11b0"
RUNNER_SHA256 = "0994f0cd91cb504f2db8478f55963813a85469807f2921e38c196248f323759e"
BRIDGE_COMMIT = "24492886922c7ac92c30ffa7d74f2d41b606a916"
BRIDGE_SHA256 = "1f3e01c6e1baf5fcab5d22e41b3c49d309b5dc5f8d2a4cf0fcf5c2fc45728ae6"
P02_SHA256 = "e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07"
SWAP_SHA256 = "d11da61e9d7bb040f3263e680f026817a513553c186ab243e184ffaf6e68cd39"

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

# No CLAIM is created if GPU is not a Tesla T4.
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("STOP_NEED_TESLA_T4_NO_SCIENCE_CLAIM")
print("TTM_GPU2048_GPU=" + torch.cuda.get_device_name(0))
drive.mount("/content/drive")
root = Path("/content/drive/MyDrive")
pack = root / "TTM" / "ttm-small-pilot" / "t4_private_pack_20261108_v1"
base = (root / "TTM" / "ttm-small-pilot"
        / "t4_science_4m_1024_a2_20261108_v1"
        / "checkpoints" / "step_1024" / "checkpoint.pt")
candidate = (root / "TTM_Small4M_EmbeddingSwap"
             / "real_token_mapping_step1024_once_20261109_v1"
             / "candidate_initialization.pt")
out = root / "TTM_Small4M_EmbeddingSwap" / RUN_ID
parent = (root / "TTM_Small4M_EmbeddingSwap" /
          "ttm_small4m_gemma2_directswap_1024x4096_t4_gpu_20261109_v1")
if not (parent / "CLAIM.json").is_file() or not (parent / "RESULT.json").is_file():
    raise RuntimeError("STOP_PARENT_GPU1024_MISSING_NO_CLAIM")
src = Path("/content/ttm_gemma2_directswap2048_source")
scratch = Path("/content/ttm_gemma2_directswap2048_scratch")

for name, path in [
    ("ORIGINAL_STEP1024", base), ("REAL_GEMMA_CANDIDATE", candidate),
    ("PRIVATE_TRAIN_VAL", pack / "T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"),
    ("PRIVATE_PACK_MANIFEST", pack / "T4_PRIVATE_PACK_MANIFEST.json")
]:
    if not path.is_file():
        raise RuntimeError("STOP_MISSING_PINNED_INPUT:" + name + ":" + str(path))
if out.exists() and not RESUME:
    raise RuntimeError("STOP_RUN_ID_ALREADY_EXISTS; use RESUME=True ONLY after interruption")
if not out.exists() and RESUME:
    raise RuntimeError("STOP_NO_CLAIM_TO_RESUME")

# Pull exact builder from an IMMUTABLE GitHub commit; it reconstructs and
# hashes the GPU2048 runner from the prior published frozen-source archive.
src.mkdir(parents=True, exist_ok=True)
builder = src / "ttm_build_directswap2048_from_locked1024.py"
url = ("https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/"
       + BUILDER_COMMIT + "/tools/embeddinggemma2_swap/" + builder.name)
if builder.exists():
    if sha(builder) != BUILDER_SHA256:
        raise RuntimeError("STOP_LOCAL_BUILDER_PIN_MISMATCH")
else:
    with urllib.request.urlopen(url, timeout=120) as stream:
        body = stream.read()
    if hashlib.sha256(body).hexdigest() != BUILDER_SHA256:
        raise RuntimeError("STOP_REMOTE_BUILDER_SHA_MISMATCH")
    with builder.open("xb") as stream:
        stream.write(body)
if sha(builder) != BUILDER_SHA256:
    raise RuntimeError("STOP_BUILDER_READBACK")

source_build = subprocess.run([sys.executable, str(builder), "--source-dir", str(src)],
                              check=False)
if source_build.returncode:
    raise RuntimeError("STOP_EXACT_SOURCE_RECONSTRUCTION_FAILED_NO_CLAIM")
for name, digest in [
    ("ttm_directswap2048_gpu_once.py", RUNNER_SHA256),
    ("p02_exact.py", P02_SHA256),
    ("ttm_embeddinggemma2_swap.py", SWAP_SHA256)
]:
    if sha(src / name) != digest:
        raise RuntimeError("STOP_GENERATED_SOURCE_SHA_MISMATCH:" + name)

# Fetch human-readable bridge source by immutable commit and SHA. No editing source after claim.
bridge = src / "ttm_directswap2048_checkpoint_bridge_20261109.py"
bridge_url = ("https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/" +
              BRIDGE_COMMIT + "/tools/embeddinggemma2_swap/" + bridge.name)
if bridge.exists():
    if sha(bridge) != BRIDGE_SHA256:
        raise RuntimeError("STOP_LOCAL_BRIDGE_SHA_MISMATCH")
else:
    with urllib.request.urlopen(bridge_url, timeout=120) as stream:
        raw = stream.read()
    if hashlib.sha256(raw).hexdigest() != BRIDGE_SHA256:
        raise RuntimeError("STOP_REMOTE_BRIDGE_SHA_MISMATCH")
    with bridge.open("xb") as f:
        f.write(raw)
if sha(bridge) != BRIDGE_SHA256:
    raise RuntimeError("STOP_BRIDGE_READBACK_SHA_MISMATCH")

cmd = [sys.executable, str(bridge),
       "--base", str(base), "--candidate", str(candidate),
       "--pack", str(pack / "T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"),
       "--manifest", str(pack / "T4_PRIVATE_PACK_MANIFEST.json"),
       "--source-dir", str(src), "--output", str(out),
       "--scratch", str(scratch), "--runner-sha256", RUNNER_SHA256,
       "--bridge-sha256", BRIDGE_SHA256, "--parent", str(parent)]
if RESUME:
    cmd.append("--resume")
print("TTM_GPU2048_RUN_ID=" + RUN_ID, flush=True)
print("TTM_GPU2048_SOURCE_COMMIT=" + BRIDGE_COMMIT, flush=True)
print("TTM_GPU2048_SCOPE=4_ARMS_INHERIT_1024_PLUS_1024_NEW_STEPS_4096_TARGETS_EACH", flush=True)
result = subprocess.run(cmd, check=False)
if result.returncode:
    raise RuntimeError("TTM_GPU2048_INCOMPLETE_OR_FAILED_EXIT_" + str(result.returncode))
print("TTM_GPU2048_DRIVE_RESULT=" + str(out / "RESULT.json"))
